# 09 · Exact asset-level coverage tests

Complements the asymptotic Kupiec test with exact binomial tests, Clopper-Pearson intervals for the coverage of each
instrument, the number of rejections expected under exact nominal coverage, and a dispersion check. It **only reads
saved bounds**; nothing from notebooks 01-08 is changed.

Keep `config/config.yaml` exactly as in the test-period run (`mode: holdout`). CPU runtime is enough (about 5-15 minutes).

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

## Locate the script and the saved bounds

In [ ]:
import shutil
from pathlib import Path
SCRIPTS = Path(PROJECT_DIR) / "scripts"
SCRIPTS.mkdir(exist_ok=True)
name = "exact_coverage_tests.py"
script = None
for loc in [SCRIPTS / name, Path(PROJECT_DIR) / "src" / "calvol" / name, Path(PROJECT_DIR) / name]:
    if loc.exists():
        if loc.parent != SCRIPTS:
            shutil.copy(loc, SCRIPTS / name)
            print(f"copied {loc.relative_to(PROJECT_DIR)} -> scripts/{name}")
        script = SCRIPTS / name
        break
assert script is not None, "Place exact_coverage_tests.py in scripts/ or src/calvol/"
assert MODE == "holdout", "Run this notebook with the same config as the test-period run (mode: holdout)"
for h in [21, 5, 1]:
    f = P.bounds / f"assets_h{h}.parquet"
    print(("OK      " if f.exists() else "MISSING ") + str(f))

## Run

In [ ]:
p = subprocess.Popen([sys.executable, str(script)], cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
assert p.returncode == 0, "script failed (see output above)"

### Consistency check
The Kupiec rejections recomputed here must equal those in the main results (`assets_h*_calibration_all.csv`).

In [ ]:
s = pd.read_csv(P.tables / "exact_coverage_summary.csv")
bad = 0
for h in [21, 5, 1]:
    f = P.tables / f"assets_h{h}_calibration_all.csv"
    if not f.exists():
        continue
    m = pd.read_csv(f).merge(s[s["h"] == h], on=["base", "method", "level"], suffixes=("_main", "_new"))
    d = m[m["kupiec_rejects_main"] != m["kupiec_rejects_new"]]
    bad += len(d)
    print(f"h={h}: {len(m)} cells compared, {len(d)} differ")
    if len(d):
        display(d[["base", "method", "level", "kupiec_rejects_main", "kupiec_rejects_new"]])
assert bad == 0, "Kupiec counts differ from the main results"

## Files to send

In [ ]:
import zipfile
z = P.results_root / "calvol_exact_coverage_results.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(P.tables.glob("exact_coverage*")):
        zf.write(f, f.relative_to(P.results_root))
        print("added", f.name)
print("Archive:", z)
if IN_COLAB:
    from google.colab import files
    files.download(str(z))